# Descriptive analysis of the ACR appropriateness criteria for abdominal/pelvic emergency department presentations

Efforts to improve CT use have focused on evidence-based decision rules and clinical practice guidelines. Among these, the American College of Radiology (ACR) Appropriateness Criteria (AC) characterize imaging modalities as usually appropriate, may be appropriate, or usually not appropriate. We determined the frequency of CT and alternative imaging ratings in ACR AC for adult patients with abdominal complaints.

## Data import and initial filtering

During October and November, 2024, a [web-scraping process](scrape.py) was used to download the structured imaging appropriateness ratings of the AC from the [ACR AC Portal](https://gravitas.acr.org/acportal) using links provided to each scenario on the portal home page. These links are stored in [acr_ac_scenarios.csv](acr_ac_scenarios.csv) (accessed Oct 2024). 

Data was accessed in accordance with the [ACR website terms and conditions]("ACR AC Terms and Conditions-20250103.txt") and the [ACR website robots.txt file](acr-2025-0103-robots.txt). 

In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
scenarios = pd.read_csv('acr_ac_scenarios.csv')
print(f"{len(scenarios)} scenarios loaded.")
scenarios.head()

3346 scenarios loaded.


,panel,scenario-id,scenario-text,scenario-url,sex,age,body-area,priority-clinical-areas
0,Breast,3110141,"Axillary adenopathy, breast cancer suspected",https://gravitas.acr.org/ACPortal/GetDataForOneScenario?senarioId=3209,Male Only,18 - 150,Breast,NaN
1,Breast,3194817,"Axillary lump, palpable, new, bilateral, initial axilla imaging",https://gravitas.acr.org/ACPortal/GetDataForOneScenario?senarioId=5930,Female Only,18 - 150,Breast,NaN
2,Breast,3194815,"Axillary lump, palpable, new, unilateral, initial axilla imaging",https://gravitas.acr.org/ACPortal/GetDataForOneScenario?senarioId=5929,Female Only,18 - 150,Breast,NaN
3,Breast,3194837,"Axillary node, suspicious on imaging, not mammo or US, next imaging study",https://gravitas.acr.org/ACPortal/GetDataForOneScenario?senarioId=5940,Female Only,18 - 150,Breast,NaN
4,Breast,3194835,"Axillary node, suspicious on mammo or US, next imaging study",https://gravitas.acr.org/ACPortal/GetDataForOneScenario?senarioId=5939,Female Only,18 - 150,Breast,NaN


### Filter criteria

Scenarios are filtered to abdominal and pelvic complaints that are applicable to Emergency Department presentations, with exclusions. 

- Limit first by body area as defined in the ACR AC metadata.
- Exclude scenarios applicable to pediatrics, as the imaging decisions in these patients is significantly different than the adult population
- Exclude scenarios applicable to pregnant patients for the same reason.
- Remove scenarios related to interventional radiology
- Exclude scenarios related to cancer screening, staging, or pre-operative planning as these are not applicable to the emergency department.
- Added back scenarios removed above based on expert review as applicable to the ED. (1 of 2 recommends)
- Removed included scenarios based on expert review as applicable to the ED. (2 of 2 recommend) 
- Duplicates removed 

In [2]:
scenarios['body-area'].unique()

<StringArray>
[              'Breast',        'Cardiac-chest', 'Chest-abdomen-pelvis',
        'Chest-abdomen',              'Cardiac',                'Chest',
 'cardiac-chest-pelvis',       'Abdomen-pelvis',              'Abdomen',
               'Pelvis',           'Neck-chest',          'Unspecified',
          'Extremities',                'Spine',      'Lower extremity',
      'Upper extremity',         'Spine-pelvis',                 'Head',
                 'Neck',            'Head-neck',              'Maxface',
           'Head-spine']
Length: 22, dtype: str

In [5]:
filtered_scenarios = scenarios[scenarios['body-area'].isin(['Chest-abdomen', 'Chest-abdomen-pelvis', 'Abdomen-pelvis', 'Abdomen', 'Pelvis'])]
body_area_n = len(filtered_scenarios)
print(f"Number of filtered scenarios by body area: {body_area_n}")

filtered_scenarios = filtered_scenarios[filtered_scenarios['panel'] != 'Pediatric']
without_peds_n = len(filtered_scenarios)
print(f"Number of filtered scenarios without pediatric: {without_peds_n}")

filtered_scenarios = filtered_scenarios[~filtered_scenarios['scenario-text'].str.contains('pregnant', case=False, na=False)]
without_pregnant_n = len(filtered_scenarios)
print(f"Number of filtered scenarios without pregnant: {without_pregnant_n}")

filtered_scenarios = filtered_scenarios[filtered_scenarios['panel'] != 'Interventional Radiology']
without_IR_n = len(filtered_scenarios)
print(f"Number of filtered scenarios without interventional radiology: {without_IR_n}")

pattern_list = ['staging', 
                'screen', 
                'surveillance',
                'incidental finding',
                'imaging during treatment',
                'posttreatment imaging',
                'post treatment imaging',
                'preop planning',
                'pre-op planning'
                ]

pattern = '|'.join(pattern_list)
filtered_scenarios = filtered_scenarios[~filtered_scenarios['scenario-text'].str.contains(pattern, case=False, na=False)]
without_monitoring_n = len(filtered_scenarios)
print(f"Number of filtered scenarios without monitoring: {without_monitoring_n}")


Number of filtered scenarios by body area: 620
Number of filtered scenarios without pediatric: 562
Number of filtered scenarios without pregnant: 547
Number of filtered scenarios without interventional radiology: 509
Number of filtered scenarios without monitoring: 398
